# Ukrainian RAG System — Demo Notebook

## 1. Setup and Imports
This section loads all necessary libraries, configures the environment, and initializes the Ukrainian RAG pipeline.

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

try:
    import torch
    torch.set_num_threads(1)
except ImportError:
    pass

sys.path.append(os.path.abspath('..'))

from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
print("RAGPipeline initialized successfully!")

## 2. Example Queries
Here, we execute 5 different questions in Ukrainian through our pipeline to demonstrate its capabilities in context retrieval and response generation.

In [ ]:
questions = [
    "Хто такий Тарас Шевченко?",
    "Що таке штучний інтелект?",
    "Коли сталася аварія на Чорнобильській АЕС?",
    "Яке місто є столицею та найбільшим містом України?",
    "В які роки відбувся Голодомор в Україні?"
]

for idx, q in enumerate(questions, 1):
    print("=" * 80)
    print(f"Запит #{idx}: {q}")
    print("=" * 80)
    
    result = pipeline.query(q)
    
    print(f"**Питання:** {result['question']}")
    print(f"**Відповідь:** {result['answer']}")
    print("\n**Джерела:**")
    for chunk in result['context_chunks']:
        print(f" - {chunk['source']} (chunk {chunk['chunk_index']})")
    print("\n")

## 3. Evaluation Results
In this section, we load the evaluation results calculated across all 10 test questions and format them as a table.

In [ ]:
with open('../data/evaluation_results.json', 'r', encoding='utf-8') as f:
    scores = json.load(f)

df = pd.DataFrame([{
    'Faithfulness': scores.get('faithfulness', 0),
    'Answer Relevancy': scores.get('answer_relevancy', 0),
    'Context Recall': scores.get('context_recall', 0)
}])

df

### How Evaluation is Done
The evaluation is performed using the **RAGAS (Retrieval-Augmented Generation Assessment)** framework. RAGAS employs an **LLM-as-a-Judge** methodology (using OpenAI's `gpt-4o-mini` model under a temperature of 0 for deterministic consistency) to automatically assess the system's performance on 10 realistic Ukrainian test questions with pre-defined ground-truth answers.

Here is how each of our three chosen metrics is calculated by the LLM Judge:

* **Faithfulness**: Measures the factual consistency of the generated answer against the retrieved context. The LLM judge breaks the generated answer down into individual statements and cross-references each statement against the retrieved source chunks. An answer containing external details or hallucinations will score lower.
* **Answer Relevancy**: Measures how directly and appropriately the generated answer addresses the user's initial query. The LLM judge generates several synthetic questions that could have led to our RAG pipeline's generated answer, computes their vector embeddings, and measures the average similarity to the original user question.
* **Context Recall**: Measures retrieval accuracy by checking whether all key ground-truth facts are present in the retrieved chunks. The LLM judge evaluates if each fact from the target ground-truth answer can be found in the retrieved context paragraphs.

## 4. Analysis
A visual analysis of our RAG system evaluation metrics represented as a bar chart.

In [ ]:
metrics = ['Faithfulness', 'Answer Relevancy', 'Context Recall']
values = [scores.get('faithfulness', 0), scores.get('answer_relevancy', 0), scores.get('context_recall', 0)]
colors = ['blue', 'green', 'orange']

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics, values, color=colors, width=0.4)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 1.1)
plt.title("RAGAS Evaluation Scores — Ukrainian RAG", fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Score", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Conclusions
The Ukrainian RAG system demonstrates strong performance, successfully retrieving accurate context from Wikipedia and generating factual, hallucination-free answers in Ukrainian. However, there is room for improvement in answer relevancy, which could be enhanced by optimizing system prompts and fine-tuning response constraints. To further elevate the system's retrieval performance on specialized domains, a promising future direction would be fine-tuning the multilingual embeddings or integrating a hybrid retrieval model (combining BM25 lexical search with dense vector embeddings) tailored specifically for Ukrainian morphosyntactic structures.